In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from pypdf import PdfReader

Chunking

In [ ]:

from langchain_community.document_loaders import PyPDFLoader,  PyMuPDFLoader
import re
import unicodedata

loader = PyMuPDFLoader(file_path="document.pdf")
pages = loader.load()


data = loader.load()
def clean(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)          # "aﬃliates" -> "affiliates"
    text = re.sub(r"Copyright © 2026 Amazon Web Services.*?All rights reserved\.", "", text, flags=re.S)
    text = text.replace("AWS Prescriptive Guidance", "")
    text = text.replace("Writing best practices to optimize RAG applications", "")
    text = re.sub(r"\.{2,}", "", text)                   # TOC dot leaders
    text = re.sub(r"\n\s*\n+", "\n", text)               # blank lines
    return text.strip()

for p in pages:
    p.page_content = clean(p.page_content)
pages = [p for p in pages if p.page_content] 


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,chunk_overlap=200
)
chunks = splitter.split_documents(pages)
chunks

In [ ]:
#embedding models 
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = FAISS.from_documents(chunks, embeddings)


In [ ]:
vectorstore.save_local("faiss_index")

In [ ]:
loaded_vectorstore = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)

In [ ]:
loaded_vectorstore

In [ ]:
 # 1. Index (once)
vectorstore = FAISS.from_documents(chunks, embeddings)

  # 2. Retrieve
query = "What is the refund policy?"
docs = vectorstore.similarity_search(query, k=4)   # top 4 chunks
for d in docs:
  print(d.page_content, d.metadata)

#With scores, so you can see how close each match is:

results = vectorstore.similarity_search_with_score(query, k=4)
for doc, score in results:
  print(score, doc.page_content[:100])

context = "\n\n".join(d.page_content for d in docs)
prompt = f"Answer using only this context:\n{context}\n\nQuestion: {query}"

In [ ]:
results = vectorstore.similarity_search_with_score(query, k=4)
for doc, score in results:
    print(score, doc.page_content[:100])

In [ ]:
def rag_prompt(query):
    docs = vectorstore.similarity_search(query, k=4)   # top 4 chunks
    for d in docs:
        print(d.page_content, d.metadata)
    context = "\n\n".join(d.page_content for d in docs)
    prompt = f"Answer using only this context:\n{context}\n\nQuestion: {query}"
    return prompt


In [ ]:

from langchain_community.document_loaders import PyPDFLoader,  PyMuPDFLoader
import re
import unicodedata

loader = PyMuPDFLoader(file_path="document.pdf")
pages = loader.load()


data = loader.load()
def clean(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)          # "aﬃliates" -> "affiliates"
    text = re.sub(r"Copyright © 2026 Amazon Web Services.*?All rights reserved\.", "", text, flags=re.S)
    text = text.replace("AWS Prescriptive Guidance", "")
    text = text.replace("Writing best practices to optimize RAG applications", "")
    text = re.sub(r"\.{2,}", "", text)                   # TOC dot leaders
    text = re.sub(r"\n\s*\n+", "\n", text)               # blank lines
    return text.strip()

for p in pages:
    p.page_content = clean(p.page_content)
pages = [p for p in pages if p.page_content] 

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def rag_func(chunk_size,chunk_overlap,vector_name,pages):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,chunk_overlap=chunk_overlap
    )
    chunks = splitter.split_documents(pages)

    #embedding models 
    from langchain_huggingface import HuggingFaceEmbeddings
    from langchain_community.vectorstores import FAISS

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-mpnet-base-v2",
        encode_kwargs={"normalize_embeddings": True},
    )

    vectorstore = FAISS.from_documents(chunks, embeddings)
    vectorstore.save_local(vector_name)
    return vectorstore

In [ ]:
rag_func(500,100,"faiss_index500",pages)